# Stream Parquet through Datashader in the browser

This notebook writes a two-million-row Parquet file in bounded chunks, then reads only the required columns as Arrow record batches. The complete table is never materialized as one Pandas DataFrame. Everything executes in this browser tab.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import datashader.transfer_functions as tf

from datashader_browser import Progress, StreamingCanvas, iter_parquet_batches

In [ ]:
path = Path('browser-points.parquet')
total_rows = 2_000_000
write_batch_size = 100_000
rng = np.random.default_rng(42)

schema = pa.schema([('x', pa.float64()), ('y', pa.float64()), ('value', pa.float64())])
with pq.ParquetWriter(path, schema, compression='zstd') as writer:
    for start in range(0, total_rows, write_batch_size):
        size = min(write_batch_size, total_rows - start)
        frame = pd.DataFrame({
            'x': rng.normal(size=size),
            'y': rng.normal(size=size),
            'value': rng.normal(20, 3, size=size),
        })
        writer.write_table(pa.Table.from_pandas(frame, schema=schema, preserve_index=False))

print(f'{path.stat().st_size / 1_000_000:.1f} MB written without building one complete table')

In [ ]:
updates = []
canvas = StreamingCanvas(
    plot_width=800,
    plot_height=500,
    x_range=(-4, 4),
    y_range=(-4, 4),
)
aggregate = canvas.points(
    iter_parquet_batches(path, batch_size=100_000, columns=['x', 'y']),
    'x',
    'y',
    on_progress=updates.append,
)
aggregate.attrs

In [ ]:
assert aggregate.attrs['source_rows'] == total_rows
assert updates[-1] == Progress(chunks=20, rows=total_rows)
tf.shade(aggregate, cmap=['#101820', '#00a6a6', '#f2aa4c'], how='eq_hist').to_pil()